# Исследование моделей классификации мобильных устройств

Блокнот содержит эксперименты по построению и оптимизации модели классификации ценового диапазона мобильных устройств.

## Содержание
1. Импорты и настройки
2. Загрузка и подготовка данных
3. Baseline модель
4. Feature Engineering (sklearn)
5. Feature Selection (RFE)
6. Production модель

## 1. Импорты и настройки

In [40]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, TargetEncoder, QuantileTransformer, SplineTransformer, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import precision_score, recall_score, f1_score

from mlxtend.feature_selection import SequentialFeatureSelector
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

import mlflow
from mlflow.models import infer_signature

import optuna

warnings.filterwarnings('ignore')

### Настройки MLflow

In [41]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

registry_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"
tracking_uri = f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(registry_uri)

EXPERIMENT_NAME = "estate_project"
REGISTRY_MODEL_NAME = "estate_model_rf"
REQ_FILE = '../requirements.txt'

## 2. Загрузка и подготовка данных

In [42]:
df = pd.read_pickle('../data/clean_data.pkl')
df = df.rename(columns={'price_range': 'target'})
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1437 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   battery_power  1437 non-null   int64   
 1   blue           1437 non-null   category
 2   clock_speed    1437 non-null   float16 
 3   dual_sim       1437 non-null   category
 4   fc             1437 non-null   int8    
 5   four_g         1437 non-null   category
 6   int_memory     1437 non-null   int8    
 7   m_dep          1437 non-null   float16 
 8   mobile_wt      1437 non-null   int64   
 9   n_cores        1437 non-null   int8    
 10  pc             1437 non-null   int8    
 11  px_height      1437 non-null   int64   
 12  px_width       1437 non-null   int64   
 13  ram            1437 non-null   int64   
 14  sc_h           1437 non-null   int8    
 15  sc_w           1437 non-null   int8    
 16  talk_time      1437 non-null   int8    
 17  three_g        1437 non-null   categor

### Разделение на обучающую и тестовую выборки

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('target', axis=1), 
    df['target'], 
    test_size=0.25, 
    random_state=2
)

### Определение типов признаков

In [44]:
cat_features = X_train.select_dtypes(include=['category', 'object']).columns.to_list()
num_features = X_train.select_dtypes(include=['number']).columns.to_list()

print(f"Категориальные признаки ({len(cat_features)}): {cat_features}")
print(f"Числовые признаки ({len(num_features)}): {num_features}")

Категориальные признаки (6): ['blue', 'dual_sim', 'four_g', 'three_g', 'touch_screen', 'wifi']
Числовые признаки (14): ['battery_power', 'clock_speed', 'fc', 'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height', 'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time']


## 3. Baseline модель

Простая модель Random Forest с минимальной предобработкой:
- StandardScaler для числовых признаков
- TargetEncoder для категориальных признаков


In [45]:
s_scaler = StandardScaler()
l_encoder = TargetEncoder()
classifier = RandomForestClassifier(random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),
        ('cat', l_encoder, cat_features),
    ],
    remainder='drop'
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', classifier)
])

pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['battery_power',
                                                   'clock_speed', 'fc',
                                                   'int_memory', 'm_dep',
                                                   'mobile_wt', 'n_cores', 'pc',
                                                   'px_height', 'px_width',
                                                   'ram', 'sc_h', 'sc_w',
                                                   'talk_time']),
                                                 ('cat', TargetEncoder(),
                                                  ['blue', 'dual_sim', 'four_g',
                                                   'three_g', 'touch_screen',
                                                   'wifi'])])),
                ('model', RandomForestClassifier(random_state=42))])

### Оценка качества baseline модели


In [46]:
predictions = pipeline.predict(X_test)

metrics = {
    "precision": precision_score(y_test, predictions, average='weighted'),
    "recall": recall_score(y_test, predictions, average='weighted'),
    "f1": f1_score(y_test, predictions, average='weighted')
}

print("Baseline модель:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")


Baseline модель:
  precision: 0.8310
  recall: 0.8250
  f1: 0.8239


### Подготовка артефактов для MLflow


In [47]:
signature = infer_signature(model_input=X_train.head(5))
input_example = X_train.head(5)


### Логирование baseline модели в MLflow


In [48]:
# Создаем эксперимент если он не существует
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Создан новый эксперимент: {EXPERIMENT_NAME}")
else:
    experiment_id = experiment.experiment_id
    print(f"Используем существующий эксперимент: {EXPERIMENT_NAME}")

with mlflow.start_run(run_name='baseline_model', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=REQ_FILE
    )
    mlflow.log_metrics(metrics)
    mlflow.log_params(pipeline.get_params())

print(f"Baseline модель залогирована. Run ID: {run_id}")


Используем существующий эксперимент: estate_project


2025/11/11 22:24:08 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


2025/11/11 22:24:08 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/11 22:24:08 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_model at: http://127.0.0.1:5000/#/experiments/1/runs/736ec0f3eb41473d949af68723d1b593.
2025/11/11 22:24:08 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Baseline модель залогирована. Run ID: 736ec0f3eb41473d949af68723d1b593


## 4. Feature Engineering (sklearn)

Создание новых признаков с использованием sklearn transformers:
- PolynomialFeatures для m_dep и battery_power (степень 2)
- QuantileTransformer для всех числовых признаков
- SplineTransformer для px_height

In [49]:
pf = PolynomialFeatures(degree=2)
qt = QuantileTransformer()
sp = SplineTransformer(n_knots=3, degree=3)

pf_pipeline = Pipeline(steps=[
    ('poly', pf),
    ('scale', StandardScaler())
])

preprocessor_sklearn = ColumnTransformer(
    transformers=[
        ('num', s_scaler, num_features),
        ('cat', l_encoder, cat_features),
        ('quantile', qt, num_features),
        ('poly', pf_pipeline, ['m_dep', 'battery_power']),
        ('spline', sp, ['px_height'])
    ],
    remainder='drop'
)

pipeline_sklearn = Pipeline(steps=[
    ('transform', preprocessor_sklearn),
    ('model', classifier)
])

print("Обучение модели с sklearn FE...")
model_sklearn = pipeline_sklearn.fit(X_train, y_train)
print("Обучение завершено")

Обучение модели с sklearn FE...
Обучение завершено


### Оценка модели с sklearn feature engineering

In [50]:
predictions = model_sklearn.predict(X_test)

metrics = {
    "precision": precision_score(y_test, predictions, average='weighted'),
    "recall": recall_score(y_test, predictions, average='weighted'),
    "f1": f1_score(y_test, predictions, average='weighted')
}

print("Модель с sklearn feature engineering:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

# Используем experiment_id созданный в ячейке с baseline моделью

with mlflow.start_run(run_name='fe_sklearn', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        model_sklearn,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=REQ_FILE
    )
    mlflow.log_metrics(metrics)
    mlflow.log_params(model_sklearn.get_params())

print(f"Модель sklearn FE залогирована. Run ID: {run_id}")

2025/11/11 22:24:08 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Модель с sklearn feature engineering:
  precision: 0.8570
  recall: 0.8500
  f1: 0.8517


2025/11/11 22:24:09 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/11 22:24:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run fe_sklearn at: http://127.0.0.1:5000/#/experiments/1/runs/70b1ecece01d445c971c3f4e8438c0ac.
2025/11/11 22:24:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Модель sklearn FE залогирована. Run ID: 70b1ecece01d445c971c3f4e8438c0ac


## 5. Feature Selection (RFE)

Recursive Feature Elimination (RFE) - метод отбора признаков, который рекурсивно удаляет наименее важные признаки.
Применим RFE к признакам, созданным с помощью sklearn transformers.


### RFE на sklearn признаках

In [51]:
X_train_sklearn_raw = preprocessor_sklearn.fit_transform(X_train, y_train)
X_train_sklearn = pd.DataFrame(X_train_sklearn_raw, columns=preprocessor_sklearn.get_feature_names_out())

print(f"Признаков после sklearn FE: {X_train_sklearn.shape[1]}")

rfe_skl_selector = RFE(estimator=classifier, n_features_to_select=12, step=0.2)
X_train_skl_rfe = rfe_skl_selector.fit_transform(X_train_sklearn, y_train)

rfe_cols = rfe_skl_selector.get_feature_names_out().tolist()
rfe_idx = rfe_skl_selector.support_

print(f"Отобрано признаков: {len(rfe_cols)}")
print(f"Признаки: {rfe_cols}")

Признаков после sklearn FE: 63
Отобрано признаков: 12
Признаки: ['num__battery_power', 'num__px_height', 'num__px_width', 'num__ram', 'quantile__battery_power', 'quantile__px_height', 'quantile__px_width', 'quantile__ram', 'poly__battery_power', 'poly__battery_power^2', 'spline__px_height_sp_1', 'spline__px_height_sp_3']


In [52]:
with open('rfe_skl_idx.txt', 'w+') as f:
    f.write(str(rfe_idx))
with open('rfe_skl_cols.txt', 'w+') as f:
    f.write(str(rfe_cols))

print("Файлы с отобранными признаками сохранены")

Файлы с отобранными признаками сохранены


In [53]:
class ColumnExtractor(object):
    """Экстрактор столбцов по индексам для использования в Pipeline"""
    def __init__(self, cols):
        self.cols = cols

    def transform(self, X):
        return X[:, self.cols]
    
    def fit(self, X, y=None):
        return self

In [54]:
rfe_skl_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn),
    ('rfe_extractor', ColumnExtractor(rfe_idx)),
    ('model', classifier)
])

rfe_skl_pipeline.fit(X_train, y_train)

predictions_skl = rfe_skl_pipeline.predict(X_test)

metrics = {
    "precision": precision_score(y_test, predictions_skl, average='weighted'),
    "recall": recall_score(y_test, predictions_skl, average='weighted'),
    "f1": f1_score(y_test, predictions_skl, average='weighted')
}

print("Модель с RFE на sklearn признаках:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

# Используем experiment_id созданный в ячейке с baseline моделью

with mlflow.start_run(run_name='rfe_skl_feature_selection', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        rfe_skl_pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=REQ_FILE
    )
    mlflow.log_metrics(metrics)
    mlflow.log_artifact('rfe_skl_cols.txt')
    mlflow.log_artifact('rfe_skl_idx.txt')
    mlflow.log_params(rfe_skl_pipeline.get_params())

print(f"Модель RFE sklearn залогирована. Run ID: {run_id}")

2025/11/11 22:24:10 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.


Модель с RFE на sklearn признаках:
  precision: 0.9125
  recall: 0.9111
  f1: 0.9116


2025/11/11 22:24:10 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/11 22:24:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run rfe_skl_feature_selection at: http://127.0.0.1:5000/#/experiments/1/runs/e9ee43d7d71746ea9f76508cfb829595.
2025/11/11 22:24:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.


Модель RFE sklearn залогирована. Run ID: e9ee43d7d71746ea9f76508cfb829595


## 6. Production модель

Обучение лучшей модели и регистрация для продакшена.

**Используем:** RFE на sklearn признаках (из секции 5) как лучшую модель.
**Обучение:** на тренировочной выборке (X_train, y_train)
**Валидация:** на тестовой выборке (X_test, y_test) для получения реальных метрик

In [ ]:
# Используем лучшую модель: RFE на sklearn признаках
production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_sklearn),
    ('rfe_extractor', ColumnExtractor(rfe_idx)),
    ('model', classifier)
])

print("Обучение production модели на тренировочной выборке...")
production_pipeline.fit(X_train, y_train)
print("Обучение завершено")

Обучение production модели на тренировочной выборке...
✅ Обучение завершено


In [56]:
predictions_prod = production_pipeline.predict(X_test)

metrics = {
    "precision": precision_score(y_test, predictions_prod, average='weighted'),
    "recall": recall_score(y_test, predictions_prod, average='weighted'),
    "f1": f1_score(y_test, predictions_prod, average='weighted')
}

print("Production модель:")
for metric_name, metric_value in metrics.items():
    print(f"  {metric_name}: {metric_value:.4f}")

Production модель:
  precision: 0.9125
  recall: 0.9111
  f1: 0.9116


### Регистрация production модели в MLflow

In [57]:
# Используем experiment_id созданный в ячейке с baseline моделью

with mlflow.start_run(run_name='production_model', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.sklearn.log_model(
        production_pipeline,
        artifact_path="models",
        signature=signature,
        input_example=input_example,
        pip_requirements=REQ_FILE,
        registered_model_name=REGISTRY_MODEL_NAME
    )
    mlflow.log_metrics(metrics)
    mlflow.log_params(production_pipeline.get_params())

print(f"\nProduction модель залогирована и зарегистрирована!")
print(f"Run ID: {run_id}")
print(f"Модель зарегистрирована как: {REGISTRY_MODEL_NAME}")

2025/11/11 22:24:10 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
Successfully registered model 'estate_model_rf'.
2025/11/11 22:24:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: estate_model_rf, version 1
Created version '1' of model 'estate_model_rf'.


2025/11/11 22:24:10 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - bokeh (current: 3.4.0, required: bokeh==3.5.2)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025/11/11 22:24:10 INFO mlflow.tracking._tracking_service.client: 🏃 View run production_model at: http://127.0.0.1:5000/#/experiments/1/runs/c712bbdc90bf489d8e80f24390a4c4c4.
2025/11/11 22:24:10 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1.



Production модель залогирована и зарегистрирована!
Run ID: c712bbdc90bf489d8e80f24390a4c4c4
Модель зарегистрирована как: estate_model_rf
